# NB05 — Deep Tabular Comparators: TabNet and FT-Transformer (FIXED)

GPU-oriented evaluation of TabNet and a compact FT-Transformer-style model.
Outer stratified 5-fold CV is repeated for seeds 2026/2027/2028. Within each outer-training fold, a stratified validation subset is used exclusively for early stopping; the outer test fold remains untouched.

This version adds explicit GPU reporting, deterministic seed handling, per-fold checkpoints, class probabilities, log loss, fit time, and a reproducibility log.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

!pip -q install pytorch-tabnet


In [ ]:
import time, math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score, log_loss
from pytorch_tabnet.tab_model import TabNetClassifier

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA not available. Switch Colab runtime to GPU (L4 recommended) for NB05.")


In [ ]:

OUT=RESULTS/"NB05_DEEP_TABULAR"; OUT.mkdir(exist_ok=True)
TAB=TABLES/"NB05_DEEP_TABULAR"; TAB.mkdir(exist_ok=True)
MOD=MODELS/"NB05_DEEP_TABULAR"; MOD.mkdir(exist_ok=True)

df=pd.read_excel(DATASET).rename(columns={"AspectRation":"AspectRatio"})
X=df.drop(columns=[TARGET]).astype(np.float32)
le=LabelEncoder(); y=pd.Series(le.fit_transform(df[TARGET].astype(str)),index=df.index)
classes=le.classes_


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def eval_metrics(y_true,y_pred,y_prob=None):
    d={
        "f1_macro":f1_score(y_true,y_pred,average="macro"),
        "accuracy":accuracy_score(y_true,y_pred),
        "balanced_accuracy":balanced_accuracy_score(y_true,y_pred),
        "mcc":matthews_corrcoef(y_true,y_pred),
        "kappa":cohen_kappa_score(y_true,y_pred),
    }
    if y_prob is not None:
        d["log_loss"]=log_loss(y_true,y_prob,labels=np.arange(len(classes)))
    return d

# Compact FT-Transformer-style model for continuous tabular features.
class FTTransformer(nn.Module):
    def __init__(self,n_features,n_classes,d_token=64,n_heads=4,n_layers=2,dropout=0.1):
        super().__init__()
        self.weight=nn.Parameter(torch.randn(n_features,d_token)*0.02)
        self.bias=nn.Parameter(torch.zeros(n_features,d_token))
        self.cls=nn.Parameter(torch.zeros(1,1,d_token))
        layer=nn.TransformerEncoderLayer(
            d_model=d_token,nhead=n_heads,dim_feedforward=d_token*4,
            dropout=dropout,batch_first=True,activation="gelu",norm_first=True
        )
        self.encoder=nn.TransformerEncoder(layer,num_layers=n_layers)
        self.head=nn.Sequential(nn.LayerNorm(d_token),nn.Linear(d_token,n_classes))
    def forward(self,x):
        tok=x.unsqueeze(-1)*self.weight.unsqueeze(0)+self.bias.unsqueeze(0)
        cls=self.cls.expand(x.size(0),-1,-1)
        z=self.encoder(torch.cat([cls,tok],dim=1))
        return self.head(z[:,0])

class ArrDS(Dataset):
    def __init__(self,X,y=None):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=None if y is None else torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        return self.X[i] if self.y is None else (self.X[i],self.y[i])

def fit_ft(Xtr,ytr,Xv,yv,n_classes,seed,max_epochs=200,patience=20):
    seed_everything(seed)
    model=FTTransformer(Xtr.shape[1],n_classes).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    loss_fn=nn.CrossEntropyLoss()
    gen=torch.Generator(); gen.manual_seed(seed)
    tr_loader=DataLoader(ArrDS(Xtr,ytr),batch_size=128,shuffle=True,generator=gen)
    vaX=torch.tensor(Xv,dtype=torch.float32,device=DEVICE)
    vay=np.asarray(yv)
    best=-1; best_state=None; stale=0
    for epoch in range(max_epochs):
        model.train()
        for xb,yb in tr_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=loss_fn(model(xb),yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred=model(vaX).argmax(1).cpu().numpy()
        score=f1_score(vay,pred,average="macro")
        if score>best+1e-5:
            best=score; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model,float(best),int(epoch+1)


In [ ]:
LOG=LOGS/"NB05_DEEP_TABULAR"; LOG.mkdir(exist_ok=True)
rows=[]; preds=[]
for seed in SEEDS:
    outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=seed)
    for fold,(tr,te) in enumerate(outer.split(X,y),start=1):
        fold_seed=seed+fold
        seed_everything(fold_seed)
        print(f"\n=== seed {seed} | outer fold {fold}/{OUTER_FOLDS} ===")
        Xtr_full,Xte=X.iloc[tr].values,X.iloc[te].values
        ytr_full,yte=y.iloc[tr].values,y.iloc[te].values
        tri,vi=train_test_split(np.arange(len(tr)),test_size=.15,stratify=ytr_full,random_state=fold_seed)
        scaler=StandardScaler().fit(Xtr_full[tri])
        Xtr=scaler.transform(Xtr_full[tri]).astype(np.float32)
        Xv=scaler.transform(Xtr_full[vi]).astype(np.float32)
        Xtest=scaler.transform(Xte).astype(np.float32)
        ytr=ytr_full[tri]; yv=ytr_full[vi]

        # TabNet
        t0=time.time()
        tab=TabNetClassifier(seed=fold_seed,verbose=0,device_name=DEVICE)
        tab.fit(Xtr,ytr,eval_set=[(Xv,yv)],eval_metric=["balanced_accuracy"],
                max_epochs=200,patience=20,batch_size=256,virtual_batch_size=128)
        prob=tab.predict_proba(Xtest)
        pp=prob.argmax(axis=1)
        met=eval_metrics(yte,pp,prob); met.update({
            "model":"TabNet","seed":seed,"outer_fold":fold,"n_test":len(te),
            "fit_seconds":time.time()-t0,"epochs":int(getattr(tab,"best_epoch",-1))+1})
        rows.append(met)
        for j,rid in enumerate(te):
            rec={"row_id":int(rid),"y_true":classes[int(yte[j])],"y_pred":classes[int(pp[j])],
                 "model":"TabNet","seed":seed,"outer_fold":fold}
            for k,cls in enumerate(classes): rec[f"prob_{cls}"]=float(prob[j,k])
            preds.append(rec)
        pd.DataFrame(rows).to_csv(OUT/"deep_tabular_metrics_CHECKPOINT.csv",index=False)
        pd.DataFrame(preds).to_csv(OUT/"deep_tabular_oof_predictions_CHECKPOINT.csv",index=False)
        print(f"TabNet done | Macro-F1={met['f1_macro']:.5f}")

        # FT-Transformer
        t0=time.time()
        ft,best_val,n_ep=fit_ft(Xtr,ytr,Xv,yv,len(classes),fold_seed)
        ft.eval()
        with torch.no_grad():
            logits=ft(torch.tensor(Xtest,dtype=torch.float32,device=DEVICE))
            prob=torch.softmax(logits,dim=1).cpu().numpy()
            pp=prob.argmax(axis=1)
        met=eval_metrics(yte,pp,prob); met.update({
            "model":"FTTransformer","seed":seed,"outer_fold":fold,"n_test":len(te),
            "fit_seconds":time.time()-t0,"best_val_macro_f1":best_val,"epochs":n_ep})
        rows.append(met)
        for j,rid in enumerate(te):
            rec={"row_id":int(rid),"y_true":classes[int(yte[j])],"y_pred":classes[int(pp[j])],
                 "model":"FTTransformer","seed":seed,"outer_fold":fold}
            for k,cls in enumerate(classes): rec[f"prob_{cls}"]=float(prob[j,k])
            preds.append(rec)
        pd.DataFrame(rows).to_csv(OUT/"deep_tabular_metrics_CHECKPOINT.csv",index=False)
        pd.DataFrame(preds).to_csv(OUT/"deep_tabular_oof_predictions_CHECKPOINT.csv",index=False)
        print(f"FTTransformer done | Macro-F1={met['f1_macro']:.5f} | epochs={n_ep}")

metrics=pd.DataFrame(rows); pred_df=pd.DataFrame(preds)
metrics.to_csv(OUT/"deep_tabular_metrics_by_fold.csv",index=False)
pred_df.to_csv(OUT/"deep_tabular_oof_predictions.csv",index=False)
summary=metrics.groupby("model").agg(
    mean_macro_f1=("f1_macro","mean"),sd_macro_f1=("f1_macro","std"),
    mean_accuracy=("accuracy","mean"),mean_balanced_accuracy=("balanced_accuracy","mean"),
    mean_mcc=("mcc","mean"),mean_kappa=("kappa","mean"),
    mean_log_loss=("log_loss","mean"),mean_fit_seconds=("fit_seconds","mean")
).sort_values("mean_macro_f1",ascending=False)
summary.to_csv(TAB/"deep_tabular_summary.csv")
display(summary)
run_info={
    "device":DEVICE,
    "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "classes":{int(i):str(c) for i,c in enumerate(classes)},
    "seeds":SEEDS,"outer_folds":OUTER_FOLDS,
    "validation_fraction_within_outer_training":0.15,
    "early_stopping_patience":20,
    "note":"Outer test folds untouched; validation split used only within outer-training data."
}
with open(LOG/"NB05_run_info.json","w") as f: json.dump(run_info,f,indent=2)
print("\nNB05 FIXED completed.")
print("OOF rows:",len(pred_df),"| expected:",2*len(SEEDS)*len(df))
